In [1]:
# throwaway: grabs everything model.ipynb reads from ml/datasets. run from ml/, rerun is safe (skips what's there)
import io, json, shutil, ssl, time, urllib.request, zipfile
from pathlib import Path
import certifi
import numpy as np

DEST = Path("datasets")
CTX = ssl.create_default_context(cafile=certifi.where())
UA = {"User-Agent": "Mozilla/5.0 (analog-explorer)"}
GET_BIG_DEMS = False   # LOLA 118 m + MOLA-HRSC, ~20 GB. optional, only makes training faster


def folder(name=""):
    d = DEST / name
    d.mkdir(parents=True, exist_ok=True)
    (d / ".gitignore").write_text("*\n!.gitignore\n")
    return d


def http(url, body=None, headers=None, timeout=600):
    hdr = {**UA, **({"Content-Type": "application/json"} if body else {}), **(headers or {})}
    for t in range(3):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, data=body, headers=hdr), context=CTX, timeout=timeout) as r:
                return r.read()
        except Exception:
            if t == 2:
                raise
            time.sleep(2 + 3 * t)


def get(url, path):
    if path.exists():
        print("have", path)
        return
    t, tmp = time.time(), path.with_suffix(path.suffix + ".part")
    with urllib.request.urlopen(urllib.request.Request(url, headers=UA), context=CTX, timeout=600) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f, 1 << 22)   # streamed, so the 11 GB DEM doesn't sit in memory
    tmp.rename(path)
    print(f"got  {path}  {path.stat().st_size / 1e6:.0f} MB  {time.time() - t:.0f}s")


folder()

PosixPath('datasets')

In [2]:
# small files
get("https://copernicus-dem-30m.s3.amazonaws.com/tileList.txt", folder() / "glo30_tileList.txt")
for w in ("MOON", "MARS"):
    get(f"https://asc-planetarynames-data.s3.us-west-2.amazonaws.com/{w}_nomenclature_center_pts.zip",
        folder("gazetteer") / f"{w}_nomenclature_center_pts.zip")
get("https://asc-astropedia.s3.us-west-2.amazonaws.com/Moon/Research/Craters/Ancillary/Catalog_Moon_Release_20180815_shapefile180.zip",
    folder("robbins") / "Catalog_Moon_Release_20180815_shapefile180.zip")
get("https://craters.sjrdesign.net/Catalog_Mars_Release_2020_1kmPlus_FullMorphData.csv.zip",
    folder("robbins") / "Catalog_Mars_Release_2020_1kmPlus_FullMorphData.csv.zip")
get("https://geodata.ucdavis.edu/climate/worldclim/2_1/base/wc2.1_10m_prec.zip", folder("climate") / "wc2.1_10m_prec.zip")
get("https://webservices.volcano.si.edu/geoserver/GVP-VOTW/ows?service=WFS&version=1.0.0&request=GetFeature"
    "&typeName=GVP-VOTW:Smithsonian_VOTW_Holocene_Volcanoes&outputFormat=application%2Fjson", DEST / "gvp_holocene.json")

got  datasets/glo30_tileList.txt  1 MB  1s
got  datasets/gazetteer/MOON_nomenclature_center_pts.zip  24 MB  2s
got  datasets/gazetteer/MARS_nomenclature_center_pts.zip  5 MB  1s
got  datasets/robbins/Catalog_Moon_Release_20180815_shapefile180.zip  138 MB  5s
got  datasets/robbins/Catalog_Mars_Release_2020_1kmPlus_FullMorphData.csv.zip  39 MB  2s
got  datasets/climate/wc2.1_10m_prec.zip  7 MB  1s
got  datasets/gvp_holocene.json  2 MB  1s


In [3]:
# geologic maps: the zips are 224 MB + 809 MB, so just pull the 4 layers we use out with range requests
class RangeFile(io.RawIOBase):
    def __init__(self, url):
        self.url, self.pos = url, 0
        with urllib.request.urlopen(urllib.request.Request(url, method="HEAD", headers=UA), context=CTX, timeout=60) as r:
            self.size = int(r.headers["Content-Length"])

    def readable(self): return True
    def seekable(self): return True
    def tell(self): return self.pos

    def seek(self, off, whence=0):
        self.pos = {0: off, 1: self.pos + off, 2: self.size + off}[whence]
        return self.pos

    def readinto(self, b):
        if self.pos >= self.size:
            return 0
        data = http(self.url, headers={"Range": f"bytes={self.pos}-{min(self.size, self.pos + len(b)) - 1}"})
        b[:len(data)] = data
        self.pos += len(data)
        return len(data)


MAPS = {
    "https://asc-astropedia.s3.us-west-2.amazonaws.com/Moon/Geology/Unified_Geologic_Map_of_the_Moon_GIS_v2.zip": [
        "Unified_Geologic_Map_of_the_Moon_GIS/Lunar_GIS/Shapefiles/GeoUnits",
        "Unified_Geologic_Map_of_the_Moon_GIS/Lunar_GIS/Shapefiles/Linear_Features"],
    "https://pubs.usgs.gov/sim/3292/downloads/sim3292_database.zip": [
        "SIM3292_MarsGlobalGeologicGIS_20M/SIM3292_Shapefiles/SIM3292_Global_Geology",
        "SIM3292_MarsGlobalGeologicGIS_20M/SIM3292_Shapefiles/SIM3292_Global_Structure"],
}
geo = folder("geology")
for url, stems in MAPS.items():
    todo = [(s, e) for s in stems for e in (".shp", ".dbf", ".prj") if not (geo / (s.rsplit("/", 1)[1] + e)).exists()]
    if not todo:
        print("have", url.rsplit("/", 1)[1])
        continue
    zf = zipfile.ZipFile(io.BufferedReader(RangeFile(url), buffer_size=1 << 22))
    for s, e in todo:
        (geo / (s.rsplit("/", 1)[1] + e)).write_bytes(zf.read(s + e))
        print("got ", s.rsplit("/", 1)[1] + e)

got  GeoUnits.shp
got  GeoUnits.dbf
got  GeoUnits.prj
got  Linear_Features.shp
got  Linear_Features.dbf
got  Linear_Features.prj
got  SIM3292_Global_Geology.shp
got  SIM3292_Global_Geology.dbf
got  SIM3292_Global_Geology.prj
got  SIM3292_Global_Structure.shp
got  SIM3292_Global_Structure.dbf
got  SIM3292_Global_Structure.prj


In [4]:
# CTX DTM footprint index from the USGS STAC catalogue (~90 pages, a couple of minutes)
path = folder("stac") / "ctx_dtm_index.npz"
if path.exists():
    print("have", path)
else:
    body, bbox, href = {"collections": ["mro_ctx_controlled_usgs_dtms"], "limit": 500}, [], []
    while body is not None:
        r = json.loads(http("https://stac.astrogeology.usgs.gov/api/search", json.dumps(body).encode(), timeout=180))
        for f in r["features"]:
            b, a = f["bbox"], f["assets"]
            bbox.append(b[:4] if len(b) == 4 else [b[0], b[1], b[3], b[4]])
            href.append(next(a[k]["href"] for k in ("dtm", "dem", "geoid_adjusted_dem") if k in a))
        nxt = [l for l in r.get("links", []) if l.get("rel") == "next"]
        body = nxt[0].get("body") if nxt and r["features"] else None
    np.savez_compressed(path, bbox=np.array(bbox, np.float32), href=np.array(href))
    print(f"got  {path}  {len(href)} footprints")

got  datasets/stac/ctx_dtm_index.npz  44628 footprints


In [5]:
# optional: the two global DEMs for the 32 km scale (James already has both, copying is faster)
if GET_BIG_DEMS:
    get("https://planetarymaps.usgs.gov/mosaic/Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.tif",
        folder("dem") / "Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.tif")
    get("https://planetarymaps.usgs.gov/mosaic/Mars/HRSC_MOLA_Blend/Mars_HRSC_MOLA_BlendDEM_Global_200mp_v2.tif",
        folder("dem") / "Mars_HRSC_MOLA_BlendDEM_Global_200mp_v2.tif")